[Course Home](../README.md) |
[Python Lab](../python/08_fisher_information_and_information_geometry.ipynb) |
[Prev: Ch 07: MaxEnt](07_the_principle_of_maximum_entropy_maxent.ipynb) |
[Next: Ch 09: Rate-Distortion & VAEs](09_rate_distortion_and_information_bottleneck.ipynb)

*Part IV: MaxEnt, Information Geometry & Decision Theory — Chapter 08 of 11*

---

# Chapter 8: The Curvature of Belief — Fisher Information & Information Geometry
### *Score Functions, The Cramér-Rao Bound, The Hessian of KL Divergence,*
### *and Natural Gradient*

---

## 1. How Does Shannon Information Connect to Statistics?

In the mid-twentieth century, two foundational theories of information
developed in parallel across the scientific landscape:

1. **Shannon Information (1948)**: Formulated in communication theory to
   quantify the fundamental limits of data compression and reliable
   transmission. It asks:
   > *"Given a known probability distribution $P(X)$, how surprising is an*
   > *observed outcome, and how many bits are required to communicate it?"*

2. **Fisher Information (1925)**: Formulated in statistical estimation theory
   by Ronald A. Fisher. It asks:
   > *"Given observed sample data $X \sim P_\theta$, how sensitive is the*
   > *distribution to changes in an unknown parameter $\theta$, and with what*
   > *precision can $\theta$ be estimated?"*

At first glance, these two quantities appear to describe entirely different
realms. Shannon information measures the global uncertainty of random
events over an alphabet of outcomes. Fisher information measures the local
sensitivity of a likelihood function with respect to coordinates on a
parameter space.

Are they merely distinct mathematical quantities sharing the same historical
name? **No. They are two complementary perspectives on the exact same**
**geometric structure.**

```
+-----------------------------------------------------------------------------+
|                      SHANNON VS. FISHER INFORMATION                         |
+-------------------------+-------------------------+-------------------------+
| Attribute               | Shannon Information     | Fisher Information      |
+-------------------------+-------------------------+-------------------------+
| Domain                  | Sample space (outcomes) | Parameter space (models)|
| Primary Object          | Distribution P(X)       | Parametric family P_theta
| Operational Question    | How many bits to encode?| How accurately to infer?|
| Fundamental Bound       | Source coding / Capacity| Cramér-Rao Lower Bound  |
| Mathematical Character  | Global expectation      | Local Riemannian metric |
+-------------------------+-------------------------+-------------------------+
```

The bridge between them is **Relative Entropy (Kullback-Leibler Divergence)**.
While Shannon entropy measures the absolute uncertainty of a single
distribution, KL divergence measures the informational separation between
two distributions. When we bring two distributions infinitesimally close
together along a parameterized family—asking how much "Shannon surprise"
is generated by nudging parameter $\theta$ to $\theta + d\theta$—the
second-order coefficient that emerges is **Fisher Information**.

Fisher information is literally the **local curvature tensor of Shannon's**
**relative entropy**. Understanding this connection unlocks the field of
**Information Geometry**, turning probability distributions into smooth
geometric manifolds where inference and learning become navigation along
geodesic paths.

---

## 2. The Score Function & Fisher Information

Let $X$ be a random variable governed by a parametric probability density or
mass function $p(x; \theta)$, where $\theta \in \Theta \subseteq \mathbb{R}^d$.
When an observation $x$ is gathered, the **log-likelihood** of parameter
$\theta$ is defined as:
$$
\ell(\theta; x) = \ln p(x; \theta)
$$

### The Score Function
The **Score Function** $S(\theta)$ is the gradient of the log-likelihood
with respect to the parameter vector $\theta$:
$$
S(\theta) = \nabla_\theta \ln p(x; \theta)
= \frac{\nabla_\theta p(x; \theta)}{p(x; \theta)}
$$

The score measures the sensitivity of the log-probability of an observed
data point to perturbations in $\theta$. If moving $\theta$ in a certain
direction dramatically increases the probability of seeing $x$, the score
points strongly in that direction.

### Theorem: The Expected Score is Always Zero
Under standard regularity conditions (allowing the interchange of
differentiation and integration):
$$
\mathbb{E}_{X \sim p_\theta}[S(\theta)] = 0
$$

**Proof:**
Expand the expectation as an integral over the sample space:
$$
\mathbb{E}_{X \sim p_\theta}[S(\theta)]
= \int p(x; \theta) \nabla_\theta \ln p(x; \theta) \, dx
$$

Applying the score identity
Applying the score identity:
$$
\nabla_\theta \ln p(x; \theta) = \frac{\nabla_\theta p(x; \theta)}{p(x; \theta)}
$$
$$
\mathbb{E}[S(\theta)]
= \int p(x; \theta) \frac{\nabla_\theta p(x; \theta)}{p(x; \theta)} \, dx
$$
$$
\mathbb{E}[S(\theta)] = \int \nabla_\theta p(x; \theta) \, dx
$$

Interchanging differentiation and integration:
$$
\mathbb{E}[S(\theta)] = \nabla_\theta \int p(x; \theta) \, dx
$$

Because $p(x; \theta)$ is a valid probability density for every $\theta$, the
integral over the entire sample space is identically 1:
$$
\mathbb{E}[S(\theta)] = \nabla_\theta (1) = 0
$$

On average, the score vector has zero mean. The data pulls $\theta$ in
different directions, but when averaged over the true distribution, all
forces perfectly cancel.

### Definition of Fisher Information
Because the expected score is zero, the variance of the score function
captures the expected magnitude of fluctuations around zero. The
**Fisher Information Matrix** $\mathcal{I}(\theta)$ is defined as the covariance
matrix of the score:
$$
\mathcal{I}(\theta) = \text{Cov}(S(\theta))
= \mathbb{E}\left[ S(\theta) S(\theta)^T \right]
$$
$$
\mathcal{I}(\theta) = \mathbb{E}\left[
  \left(\nabla_\theta \ln p(X; \theta)\right)
  \left(\nabla_\theta \ln p(X; \theta)\right)^T
\right]
$$

For a scalar parameter $\theta$, this reduces to:
$$
\mathcal{I}(\theta)
= \mathbb{E}\left[
  \left( \frac{\partial}{\partial \theta} \ln p(X; \theta) \right)^2
\right]
$$

### The Second-Derivative Identity (Hessian Form)
There is a second, equally important formulation of Fisher information:
it is the negative expectation of the Hessian matrix of the log-likelihood.
$$
\mathcal{I}(\theta) = -\mathbb{E}\left[ \nabla_\theta^2 \ln p(X; \theta) \right]
$$

**Proof (Scalar Case):**
Compute the second derivative of the log-likelihood:
$$
\frac{\partial^2}{\partial \theta^2} \ln p(x; \theta)
= \frac{\partial}{\partial \theta} \left(
  \frac{1}{p(x; \theta)} \frac{\partial p(x; \theta)}{\partial \theta}
\right)
$$

Applying the quotient rule:
$$
\frac{\partial^2}{\partial \theta^2} \ln p(x; \theta)
= -\frac{1}{p(x; \theta)^2}
  \left( \frac{\partial p(x; \theta)}{\partial \theta} \right)^2
+ \frac{1}{p(x; \theta)} \frac{\partial^2 p(x; \theta)}{\partial \theta^2}
$$

Recognizing that
Recognizing that:
$$
\frac{1}{p(x;\theta)} \frac{\partial p(x;\theta)}{\partial \theta}
= \frac{\partial \ln p(x;\theta)}{\partial \theta}
$$
$$
\frac{\partial^2}{\partial \theta^2} \ln p(x; \theta)
= -\left( \frac{\partial}{\partial \theta} \ln p(x; \theta) \right)^2
+ \frac{1}{p(x; \theta)} \frac{\partial^2 p(x; \theta)}{\partial \theta^2}
$$

Now take the expectation over $X \sim p(x; \theta)$:
$$
\mathbb{E}\left[ \frac{\partial^2}{\partial \theta^2} \ln p(X; \theta) \right]
= -\mathbb{E}\left[
  \left( \frac{\partial}{\partial \theta} \ln p(X; \theta) \right)^2
\right]
+ \int \frac{\partial^2 p(x; \theta)}{\partial \theta^2} \, dx
$$

The second integral simplifies:
$$
\int \frac{\partial^2 p(x; \theta)}{\partial \theta^2} \, dx
= \frac{\partial^2}{\partial \theta^2} \int p(x; \theta) \, dx
= \frac{\partial^2}{\partial \theta^2} (1) = 0
$$

Therefore:
$$
\mathbb{E}\left[ \frac{\partial^2}{\partial \theta^2} \ln p(X; \theta) \right]
= -\mathbb{E}\left[
  \left( \frac{\partial}{\partial \theta} \ln p(X; \theta) \right)^2
\right]
= -\mathcal{I}(\theta)
$$

Multiplying by $-1$ yields the desired identity. The Fisher Information
measures the average **negative curvature** (concavity) of the log-likelihood
surface at the true parameter value.

### Concrete Example: Gaussian Location Model
Let $X \sim \mathcal{N}(\mu, \sigma_0^2)$, where the variance $\sigma_0^2$ is
known and the mean $\mu$ is unknown. The log-likelihood for a single
observation is:
$$
\ln p(x; \mu) = -\frac{1}{2} \ln(2\pi \sigma_0^2)
- \frac{(x - \mu)^2}{2\sigma_0^2}
$$

Differentiating with respect to $\mu$:
$$
S(\mu) = \frac{\partial}{\partial \mu} \ln p(x; \mu)
= \frac{x - \mu}{\sigma_0^2}
$$

Differentiating a second time:
$$
\frac{\partial^2}{\partial \mu^2} \ln p(x; \mu) = -\frac{1}{\sigma_0^2}
$$

Taking the negative expectation:
$$
\mathcal{I}(\mu)
= -\mathbb{E}\left[ -\frac{1}{\sigma_0^2} \right]
= \frac{1}{\sigma_0^2}
$$

The Fisher information for the location parameter of a Gaussian is simply the
inverse of the noise variance: $\mathcal{I}(\mu) = 1/\sigma_0^2$. High noise
implies low information; low noise implies high information.

### The "Spring Stiffness" Mental Model
Imagine the log-likelihood function as an upside-down physical bowl, with the
parameter $\theta$ resting at the peak:

```
 HIGH FISHER INFORMATION (Stiff Spring)    LOW FISHER INFORMATION (Soft Spring)
 
           ln p(x; theta)                           ln p(x; theta)
                 *                                        ***
                / \                                    *       *
               /   \                                  /         \
              /     \                                /           \
             /       \                              /             \
            *         *                            *               *
       -----------------------               -----------------------------
               theta                                     theta
        Sharp curvature = Large I             Gentle slope = Small I
        Parameter pinned tightly              Parameter drifts easily
```

- **Stiff Spring (Large $\mathcal{I}(\theta)$)**: The likelihood curve drops
  steeply if $\theta$ deviates even slightly from its true value. The data
  constrains the parameter powerfully. An estimator will have small variance.
- **Loose Spring (Small $\mathcal{I}(\theta)$)**: The likelihood surface is
  flat. Drastic shifts in $\theta$ barely alter the probability of the data.
  The data provides almost no signal, and any estimator will be plagued by
  high variance.

---

## 3. The Cramér-Rao Lower Bound (CRLB)

Why does Fisher information matter for real-world engineering and statistics?
Because it establishes an impenetrable theoretical floor on estimation error.

Let $\hat{\theta}(X)$ be any unbiased estimator of a scalar parameter
$\theta$, meaning $\mathbb{E}[\hat{\theta}(X)] = \theta$ for all
$\theta \in \Theta$. How small can the variance $\text{Var}(\hat{\theta})$
possibly be?

### Theorem: The Cramér-Rao Inequality
For any unbiased estimator $\hat{\theta}(X)$ computed from sample $X$:
$$
\text{Var}(\hat{\theta}) \ge \frac{1}{\mathcal{I}(\theta)}
$$

For $N$ independent and identically distributed (i.i.d.) observations
$X_1, X_2, \dots, X_N$, the total Fisher information is additive:
$\mathcal{I}_N(\theta) = N \mathcal{I}(\theta)$.
Thus, the fundamental bound becomes:
$$
\text{Var}(\hat{\theta}) \ge \frac{1}{N \mathcal{I}(\theta)}
$$

No unbiased estimation algorithm—regardless of whether it uses deep neural
networks, Markov chain Monte Carlo, or quantum computing—can achieve an
error variance smaller than $\frac{1}{N \mathcal{I}(\theta)}$.

### Step-by-Step Proof via Cauchy-Schwarz
Consider two random variables: the estimation error
$U = \hat{\theta}(X) - \theta$ and the score
$V = S(\theta) = \frac{\partial}{\partial \theta} \ln p(X; \theta)$.

Recall that $\mathbb{E}[V] = \mathbb{E}[S(\theta)] = 0$. Therefore:
$$
\text{Cov}(U, V) = \mathbb{E}[U V]
= \mathbb{E}[(\hat{\theta}(X) - \theta) S(\theta)]
= \mathbb{E}[\hat{\theta}(X) S(\theta)]
$$

Express this expectation as an integral:
$$
\mathbb{E}[\hat{\theta}(X) S(\theta)]
= \int \hat{\theta}(x) \left(
  \frac{\partial \ln p(x; \theta)}{\partial \theta}
\right) p(x; \theta) \, dx
$$
$$
\mathbb{E}[\hat{\theta}(X) S(\theta)]
= \int \hat{\theta}(x) \frac{\partial p(x; \theta)}{\partial \theta} \, dx
$$

Interchanging integration and differentiation:
$$
\mathbb{E}[\hat{\theta}(X) S(\theta)]
= \frac{\partial}{\partial \theta} \int \hat{\theta}(x) p(x; \theta) \, dx
$$

Because $\hat{\theta}$ is unbiased,
$\int \hat{\theta}(x) p(x; \theta) \, dx = \mathbb{E}[\hat{\theta}] = \theta$.
Hence:
$$
\mathbb{E}[\hat{\theta}(X) S(\theta)]
= \frac{\partial}{\partial \theta} (\theta) = 1
$$

We have established that the covariance between the estimator error and the
score is exactly 1:
$$
\text{Cov}(\hat{\theta}, S(\theta)) = 1
$$

Now apply the **Cauchy-Schwarz Inequality** for random variables, which
states that for any random variables $U$ and $V$:
$$
[\text{Cov}(U, V)]^2 \le \text{Var}(U) \cdot \text{Var}(V)
$$

Substitute $U = \hat{\theta} - \theta$ and $V = S(\theta)$:
$$
1^2 \le \text{Var}(\hat{\theta}) \cdot \text{Var}(S(\theta))
$$

Recalling that $\text{Var}(S(\theta)) = \mathcal{I}(\theta)$:
$$
1 \le \text{Var}(\hat{\theta}) \cdot \mathcal{I}(\theta)
\implies \text{Var}(\hat{\theta}) \ge \frac{1}{\mathcal{I}(\theta)}
$$

This completes the proof.

### Efficiency and Maximum Likelihood Estimation
An unbiased estimator that achieves the Cramér-Rao Lower Bound with equality
is called **efficient**:
$$
\text{Var}(\hat{\theta}) = \frac{1}{N \mathcal{I}(\theta)}
$$

Equality in Cauchy-Schwarz holds if and only if $U$ is a deterministic linear
function of $V$:
$$
\hat{\theta}(x) - \theta = c(\theta) S(\theta)
$$

This condition is satisfied primarily for exponential families. However, even
when no finite-sample efficient estimator exists, **Maximum Likelihood**
**Estimation (MLE)** achieves asymptotic efficiency:
$$
\sqrt{N}\left(\hat{\theta}_{\text{MLE}} - \theta\right)
\xrightarrow{d} \mathcal{N}\left(0, \mathcal{I}(\theta)^{-1}\right)
\quad \text{as } N \to \infty
$$

As sample size grows, the MLE extracts 100% of the available Fisher
information, attaining the theoretical precision ceiling.

---

## 4. The Grand Unification: Fisher Information as KL Curvature

We now arrive at the mathematical heart of this chapter: the rigorous
unification of Shannon relative entropy and Fisher information.

Consider a parametric family of probability distributions $\{P_\theta\}$.
Suppose we start at parameter vector $\theta$ and apply an infinitesimal
displacement $d\theta$, arriving at the neighboring distribution
$P_{\theta + d\theta}$.

How far apart are these two distributions in information space?
Let us measure their distance using Kullback-Leibler divergence:
$$
D(\Delta) = D_{\text{KL}}\left(P_\theta \;\parallel\; P_{\theta + \Delta}\right)
$$
$$
D(\Delta) = \int p(x; \theta) \ln \left(
  \frac{p(x; \theta)}{p(x; \theta + \Delta)}
\right) dx
$$

Let us evaluate the Taylor series expansion of $D(\Delta)$ around $\Delta = 0$:
$$
D(\Delta) = D(0)
+ \Delta^T \left[\nabla_\Delta D(\Delta)\right]_{\Delta=0}
+ \frac{1}{2} \Delta^T \left[\nabla_\Delta^2 D(\Delta)\right]_{\Delta=0} \Delta
+ \mathcal{O}(\|\Delta\|^3)
$$

Let us compute each term individually:

### 1. The Zero-Order Term
When $\Delta = 0$, both distributions are identical:
$$
D(0) = D_{\text{KL}}(P_\theta \parallel P_\theta)
= \int p(x; \theta) \ln(1) \, dx = 0
$$

### 2. The First-Order Term (Gradient)
Compute the gradient of $D(\Delta)$ with respect to $\Delta$:
$$
\nabla_\Delta D(\Delta)
= \nabla_\Delta \int p(x; \theta) [\ln p(x; \theta)
  - \ln p(x; \theta + \Delta)] \, dx
$$
$$
\nabla_\Delta D(\Delta)
= -\int p(x; \theta) \nabla_\Delta \ln p(x; \theta + \Delta) \, dx
$$

Evaluating at $\Delta = 0$:
$$
\left[\nabla_\Delta D(\Delta)\right]_{\Delta=0}
= -\int p(x; \theta) \nabla_\theta \ln p(x; \theta) \, dx
= -\mathbb{E}_{X \sim p_\theta}[S(\theta)]
$$

As we proved in Section 2, the expected score is identically zero!
$$
\left[\nabla_\Delta D(\Delta)\right]_{\Delta=0} = -0 = 0
$$

The first-order variation vanishes completely. The KL divergence achieves its
global minimum of 0 at $\Delta = 0$, meaning the tangent space is flat.

### 3. The Second-Order Term (Hessian)
Compute the Hessian matrix of $D(\Delta)$ with respect to $\Delta$:
$$
\nabla_\Delta^2 D(\Delta)
= -\int p(x; \theta) \nabla_\Delta^2 \ln p(x; \theta + \Delta) \, dx
$$

Evaluating at $\Delta = 0$:
$$
\left[\nabla_\Delta^2 D(\Delta)\right]_{\Delta=0}
= -\int p(x; \theta) \nabla_\theta^2 \ln p(x; \theta) \, dx
$$
$$
\left[\nabla_\Delta^2 D(\Delta)\right]_{\Delta=0}
= -\mathbb{E}_{X \sim p_\theta}\left[\nabla_\theta^2 \ln p(X; \theta)\right]
$$

By the second-derivative identity of Fisher information:
$$
\left[\nabla_\Delta^2 D(\Delta)\right]_{\Delta=0} = \mathcal{I}(\theta)
$$

### The Unification Formula
Substituting the three terms back into the Taylor expansion with
$\Delta = d\theta$:
$$
D_{\text{KL}}\left(P_\theta \;\parallel\; P_{\theta + d\theta}\right)
= \frac{1}{2} d\theta^T \mathcal{I}(\theta) d\theta + \mathcal{O}(\|d\theta\|^3)
$$

```
              THE BRIDGE BETWEEN SHANNON AND FISHER INFORMATION
              
                     D_KL( P_theta || P_{theta + dtheta} )
                                       |
                                       |  Taylor series around dtheta = 0
                                       v
                       1/2 * dtheta^T * I(theta) * dtheta
                                       ^
                                       |
                          Fisher Information Matrix
                        (Riemannian Metric Tensor g_ij)
```

Furthermore, reversing the direction of KL divergence yields the same result
to second order:
$$
D_{\text{KL}}\left(P_{\theta + d\theta} \;\parallel\; P_\theta\right)
= \frac{1}{2} d\theta^T \mathcal{I}(\theta) d\theta + \mathcal{O}(\|d\theta\|^3)
$$

The symmetric divergence (Jeffreys divergence) is therefore:
$$
D_{\text{sym}}(P_\theta, P_{\theta + d\theta})
= D_{\text{KL}}(P_\theta \parallel P_{\theta+d\theta})
+ D_{\text{KL}}(P_{\theta+d\theta} \parallel P_\theta)
$$
$$
D_{\text{sym}}(P_\theta, P_{\theta + d\theta})
= d\theta^T \mathcal{I}(\theta) d\theta + \mathcal{O}(\|d\theta\|^3)
$$

Locally, Kullback-Leibler divergence is a quadratic bowl whose Hessian is
the Fisher Information Matrix. Fisher information is the engine that converts
infinitesimal parameter changes into Shannon relative entropy.

---

## 5. Information Geometry & Natural Gradients

### The Statistical Manifold and the Fisher-Rao Metric
In differential geometry, a **Riemannian manifold** is a smooth curved space
endowed with a metric tensor $g_{ij}(\theta)$ that defines the distance $ds$
between infinitesimally separated points:
$$
ds^2 = \sum_{i,j} g_{ij}(\theta) d\theta_i d\theta_j
= d\theta^T g(\theta) d\theta
$$

Consider the family of all probability distributions
$\mathcal{M} = \{P_\theta : \theta \in \Theta\}$.
Because the infinitesimal KL divergence is quadratic in $d\theta$:
$$
ds^2 = 2 D_{\text{KL}}(P_\theta \parallel P_{\theta + d\theta})
= d\theta^T \mathcal{I}(\theta) d\theta
$$

The Fisher Information Matrix acts as the **canonical Riemannian metric**
**tensor** on the manifold of probability distributions:
$g_{ij}(\theta) = \mathcal{I}_{ij}(\theta)$.
This geometry is known as the **Fisher-Rao metric**.

### Chentsov's Theorem (1982)
Why this metric and not some other matrix? **Chentsov's Theorem** proves that
the Fisher-Rao metric is the **unique** Riemannian metric on statistical
manifolds (up to a global constant scalar) that is invariant under congruent
Markov embeddings (sufficient statistics and coarse-grainings).
It is the only metric that respects the intrinsic informational structure
of probability distributions rather than arbitrary coordinate conventions.

### Why Standard Euclidean Gradient Descent Fails on Distributions
In machine learning, we frequently optimize an objective $L(\theta)$ using
standard gradient descent:
$$
\theta_{t+1} = \theta_t - \eta \nabla_\theta L(\theta_t)
$$

This update rule secretly assumes that parameter space is
**flat Euclidean space**, where the distance between $\theta$ and
$\theta + d\theta$ is measured by the standard Euclidean norm
$\|d\theta\|_2^2 = \sum d\theta_i^2$.

This assumption produces severe pathologies when applied to probability
parameters:
1. **Coordinate Dependence**: If you reparameterize your model—for example,
   substituting $\phi = \ln \theta$ or $\phi = \theta^3$—the Euclidean gradient
   vector $\nabla_\phi L$ changes direction and step size completely. The
   optimization trajectory is an artifact of the chosen coordinate system
   rather than the underlying probability distribution.
2. **Plateaus and Cliffs**: In a Gaussian model $\mathcal{N}(\mu, \sigma^2)$,
   shifting $\mu$ by $0.1$ when $\sigma = 100$ produces virtually zero
   change in distribution ($D_{\text{KL}} \approx 0.00005$). But shifting
   $\mu$ by $0.1$ when $\sigma = 0.01$ causes distributions to become
   disjoint ($D_{\text{KL}} \approx 50$). Standard gradient descent treats
   both steps as having the identical Euclidean magnitude $\|\Delta\mu\| = 0.1$!

### Shun-ichi Amari's Natural Gradient (1998)
In 1998, Japanese mathematician **Shun-ichi Amari** resolved this flaw by
defining the **Natural Gradient**.

Instead of finding the direction of steepest descent under an arbitrary
Euclidean step size $\|d\theta\|_2^2 \le \epsilon^2$, we seek the direction of
steepest descent under a **constant change in the actual probability**
**distribution**, measured by KL divergence:
$$
\min_{d\theta} L(\theta + d\theta) \quad \text{subject to }
D_{\text{KL}}(P_\theta \parallel P_{\theta + d\theta})
= \frac{1}{2} d\theta^T \mathcal{I}(\theta) d\theta \le \epsilon^2
$$

Using the first-order approximation
$L(\theta + d\theta) \approx L(\theta) + \nabla_\theta L^T d\theta$
and introducing a Lagrange multiplier $\lambda$:
$$
\mathcal{L}(d\theta, \lambda) = L(\theta) + \nabla_\theta L^T d\theta
+ \frac{\lambda}{2}\left(
  d\theta^T \mathcal{I}(\theta) d\theta - \epsilon^2
\right)
$$

Taking the derivative with respect to $d\theta$ and setting to zero:
$$
\nabla_\theta L + \lambda \mathcal{I}(\theta) d\theta = 0
\implies d\theta = -\frac{1}{\lambda} \mathcal{I}(\theta)^{-1} \nabla_\theta L
$$

The vector $\tilde{\nabla}_\theta L = \mathcal{I}(\theta)^{-1} \nabla_\theta L$
is the **Natural Gradient**.

The **Natural Gradient Descent** update rule is:
$$
\theta_{t+1} = \theta_t
- \eta \mathcal{I}(\theta_t)^{-1} \nabla_\theta L(\theta_t)
$$

```
               EUCLIDEAN GRADIENT VS. NATURAL GRADIENT
               
         Parameter Space Coordinates       Statistical Manifold (Distributions)
         ---------------------------       ------------------------------------
             (Distorted grid)                        (True geometry)
                 |                                       |
                 v                                       v
          Euclidean Gradient:                    Natural Gradient:
        Steps fixed distance in               Steps fixed KL distance in
         coordinate system                     probability space
          theta <- theta - eta*g                theta <- theta - eta*I^-1*g
```

Properties of the Natural Gradient:
- **Coordinate Invariance**: The path traced in distribution space is
  invariant to any smooth, invertible reparameterization of $\theta$.
- **Curvature Compensation**: When the likelihood surface is flat (small
  Fisher information), $\mathcal{I}^{-1}$ amplifies the step size. When the
  surface is steep (large Fisher information), $\mathcal{I}^{-1}$ dampens the
  step size.
- **Geodesic Stepping**: Each step travels along a geodesic (shortest path)
  on the statistical manifold.

---

## 6. Applied Engineering Contexts

### 1. Natural Policy Gradients in Reinforcement Learning (TRPO & PPO)
In policy-gradient reinforcement learning, an agent parameterizes a
stochastic policy $\pi_\theta(a \mid s)$ to maximize expected cumulative reward
$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$.

Standard policy gradient (REINFORCE) computes $\nabla_\theta J(\theta)$.
However, a small step in parameter space can lead to a catastrophic collapse
in policy performance if the resulting change in action probabilities is too
abrupt.

Sham Kakade (2002) introduced the **Natural Policy Gradient**:
$$
\theta_{t+1} = \theta_t + \eta \mathcal{F}^{-1} \nabla_\theta J(\theta_t)
$$
where
$\mathcal{F} = \mathbb{E}_{s \sim \rho_\pi}\left[\mathcal{I}_s(\theta)\right]$
is the expected Fisher information matrix of the policy across state space.

John Schulman et al. directly formalized this into **Trust Region Policy**
**Optimization (TRPO)** (2015) and **Proximal Policy Optimization (PPO)**
(2017), the workhorse algorithms behind modern RL and LLM reinforcement
learning from human feedback (RLHF):
$$
\max_\theta \mathbb{E}\left[
  \frac{\pi_\theta(a \mid s)}{\pi_{\theta_{\text{old}}}(a \mid s)} A(s, a)
\right]
$$
$$
\text{subject to }
\mathbb{E}\left[
  D_{\text{KL}}(\pi_{\theta_{\text{old}}} \parallel \pi_\theta)
\right] \le \delta
$$

TRPO uses conjugate gradient methods to approximate the inverse Fisher
matrix-vector product, ensuring that policy updates stay within an exact
trust region on the statistical manifold.

### 2. Parameter Estimation Limits in Radar, Sonar & Lidar
In active remote sensing (radar, sonar, autonomous vehicle lidar), a pulse
$s(t)$ is transmitted, bounces off a target, and returns with time delay $\tau$
and Doppler frequency shift $\nu$ corrupted by additive white Gaussian noise:
$$
r(t) = A s(t - \tau) e^{j 2\pi \nu t} + w(t)
$$
where $w(t) \sim \mathcal{CN}(0, N_0)$.

The Cramér-Rao Lower Bound dictates the ultimate physical resolution of any
ranging sensor:
$$
\text{Var}(\hat{\tau}) \ge \frac{1}{\mathcal{I}(\tau)}
= \frac{N_0}{2 E_s \cdot \beta_{\text{rms}}^2}
$$
where $E_s = \int |s(t)|^2 dt$ is the pulse energy, and $\beta_{\text{rms}}$ is
the **root-mean-square bandwidth** of the transmitted waveform:
$$
\beta_{\text{rms}}^2
= \frac{\int_{-\infty}^\infty (2\pi f)^2 |S(f)|^2 df}
       {\int_{-\infty}^\infty |S(f)|^2 df}
$$

This fundamental bound proves that to achieve sub-millimeter ranging precision
in automotive lidar or military radar without increasing transmitter power,
engineers must maximize signal bandwidth. This directly inspired the
invention of frequency-modulated continuous-wave (FMCW) chirp signals,
which artificially maximize $\beta_{\text{rms}}$ within safe power constraints.

### 3. Jeffreys Prior in Bayesian Inference
A central debate in Bayesian statistics is how to formulate an "uninformative"
or objective prior $p(\theta)$ when no prior knowledge exists.

Naively choosing a uniform prior $p(\theta) = c$ fails because it is not
invariant under reparameterization: if you declare $\theta$ uniform, then
$\phi = \theta^2$ has prior density
$p(\phi) = \frac{c}{2\sqrt{\phi}}$, which is highly non-uniform!

Sir Harold Jeffreys (1946) used Fisher information to create the
**Jeffreys Prior**:
$$
p(\theta) \propto \sqrt{\det \mathcal{I}(\theta)}
$$

In differential geometry, for any Riemannian manifold with metric tensor $g$,
the invariant volume element is $dV = \sqrt{\det g} \, d\theta$. Under a
change of coordinates $\phi = f(\theta)$, the Jacobian transformation of the
metric tensor perfectly cancels the Jacobian of the coordinate change:
$$
p(\phi) = p(\theta)
\left| \det \frac{\partial \theta}{\partial \phi} \right|
= \sqrt{\det \mathcal{I}(\phi)}
$$

The Jeffreys prior is the **unique coordinate-invariant prior** on the
statistical manifold. It assigns equal probability mass to regions of
equal informational volume, guaranteeing that Bayesian inference produces
identical conclusions regardless of how the model is parameterized.

---

[Course Home](../README.md) |
[Python Lab](../python/08_fisher_information_and_information_geometry.ipynb) |
[Prev: Ch 07: MaxEnt](07_the_principle_of_maximum_entropy_maxent.ipynb) |
[Next: Ch 09: Rate-Distortion & VAEs](09_rate_distortion_and_information_bottleneck.ipynb)

